In [1]:
%load_ext autoreload
%autoreload 2

# Load and validate the NF background model

This notebook performs the initial setup for the standalone background-modeling workflow. It loads Pascal Janowski's trained normalizing-flow background model and downloads the complete three-month DC4 simulated background reference.

## Setup

### Imports

Use the installed `cosipy` dependency for the NF model and the official Wasabi download utility.

In [2]:
from pathlib import Path
from cosipy.background_estimation.ml import NFBackground
from cosipy.util import fetch_wasabi_file
from astropy.io import fits


WARNING TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


WARNING FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.



### File paths

Keep the trained checkpoint in `model/` and the complete simulated background in `data/`. These paths are relative to the `notebooks/` directory.

In [3]:
model_path = Path("../model/nfbackground_v2-00.pt")
background_data_path = Path(
    "../data/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz"
)

### Load the trained NF background model

Load the version 2.00 checkpoint on CPU. This verifies that `NFBackground` can deserialize the trained model; it does not evaluate or sample the density yet.

In [4]:
nf_background = NFBackground(
    model_path,
    devices=["cpu"],
    density_compile_mode=None,
)

### Download the complete DC4 background reference

Download the complete three-month simulated DC4 background only when it is not already present locally.

In [5]:
if background_data_path.exists():
    print(f"Using existing file: {background_data_path}")
else:
    fetch_wasabi_file(
        "COSI-SMEX/DC4/Data/Backgrounds/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz",
        output=str(background_data_path),
    )

Using existing file: ../data/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits.gz


## Data Exploration

In [6]:
size_gb = background_data_path.stat().st_size / 1024**3
print(f"Compressed file size: {size_gb:.2f} GB")

Compressed file size: 14.87 GB


In [7]:
with fits.open(background_data_path, memmap=False, lazy_load_hdus=True) as hdul:
    event_header = hdul[1].header

In [8]:
print(f"Extension: {event_header.get('EXTNAME', 'unnamed')}")
print(f"Events: {event_header['NAXIS2']:,}")
print(f"Columns: {event_header['TFIELDS']}")

Extension: unnamed
Events: 168,648,544
Columns: 11


In [9]:
for index in range(1, event_header["TFIELDS"] + 1):
    print(index, event_header[f"TTYPE{index}"], event_header[f"TFORM{index}"])

1 Energies D
2 TimeTags D
3 Xpointings (glon,glat) 2D
4 Ypointings (glon,glat) 2D
5 Zpointings (glon,glat) 2D
6 Phi D
7 Chi local D
8 Psi local D
9 Distance D
10 Chi galactic D
11 Psi galactic D


### Inspect the checkpoint metadata

`density_input` is metadata stored inside the trained checkpoint. The available keys document the model configuration. In particular, `menergy_cuts` and `phi_cuts` define the valid physical domain used below. The `bins` entry describes the internal rational-quadratic spline resolution; it is not analysis binning.

In [ ]:
density_metadata = nf_background._density_input

print("Available checkpoint keys:")
print(sorted(density_metadata))

In [ ]:
print("Measured-energy cuts:", density_metadata["menergy_cuts"])
print("Phi cuts:", density_metadata["phi_cuts"])

### Generate a small NF sample

Use the midpoint of the checkpoint's time range as context and request only 100 events. This is a small interface test before producing comparison samples.

In [ ]:
import torch

sample_time = density_metadata["start_time"] + density_metadata["total_time"] / 2
time_context = torch.full((100, 1), sample_time, dtype=torch.float64)
nf_samples = nf_background.sample_density(time_context)